# 🌍 绿色低碳动态雷达 · 四维透视

> 数据源：`data/history.json`（62 天累积窗口） ｜ 生成：2026-08-19
> 环境：Python 3.13 + pandas + plotly + jieba
> 操作：逐格执行（Shift+Enter），或顶部 Run All 一次跑完
>
> **Positron 提示**：代码块输出表格后，点表格右上角的小格子图标，可打开 Data Explorer 交互式翻表~

## 一、数据加载

从 `../data/history.json` 读取 62 天累积数据，转成 DataFrame。

In [1]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import jieba

DATA_DIR = Path("../data")
with open(DATA_DIR / "history.json", encoding="utf-8") as f:
    hist = json.load(f)

df = pd.DataFrame(hist["items"])
print(f"共 {len(df)} 条 | 窗口 {hist['window_days']} 天 | 生成于 {hist['generated_at']}")

共 504 条 | 窗口 62 天 | 生成于 2026-08-19T02:32:13.560035Z


In [2]:
# 混合时区（+08:00 / Z）统一转北京时间，再拆出日期
df["dt"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce").dt.tz_convert("Asia/Shanghai")
df["date"] = df["dt"].dt.date

# score_breakdown 拆成独立列（source 是"来源权威分"→src_score；people 是"人物分"，防与顶层人物名单字段撞名 →people_score）
bd = df["score_breakdown"].apply(pd.Series).rename(columns={"source": "src_score", "people": "people_score"})
df = pd.concat([df.drop(columns=["score_breakdown"]), bd], axis=1)

df[["title", "dimension", "region", "score", "score_level",
    "src_score", "strength", "topic", "freshness", "date", "source"]].head()

,title,dimension,region,score,score_level,src_score,strength,topic,freshness,date,source
0,天然气库存跌至新低，欧洲能源危机“凛冬将至”？,行业,中国,55,B,13,20,12,10,2026-08-19,中国能源报
1,全球炼油能力恢复或支撑市场回暖,行业,中国,49,C,13,20,6,10,2026-08-19,中国能源报
2,史上最大IPO快了，AI浪潮能“浪”到何时？,金融,中国,71,A,13,30,18,10,2026-08-19,中国能源报
3,山东：支持周边新能源资源条件较好的零碳园区等开展多用户绿电直连 引导重点用能企业通过自发自用...,行业,中国,66,B,18,20,18,10,2026-08-19,碳道
4,英报告：中国绿色产业发展助力全球低碳转型,行业,中国,66,B,18,20,18,10,2026-08-19,碳道


## 二、总览仪表盘

四维分布 / 评分等级 / 区域版图，一眼看清 62 天数据全貌。

In [3]:
fig1 = px.pie(df, names="dimension", title="四维分布（政府/行业/金融/AI）",
             hole=0.45, category_orders={"dimension": ["政府", "行业", "金融", "AI"]})
fig1.update_traces(textinfo="value+percent")
fig1.show()

In [4]:
fig2 = px.histogram(df, x="score", color="score_level", nbins=20,
                   title="评分分布（S≥85 / A≥70 / B≥55 / C≥40 / D）")
fig2.show()

In [5]:
fig3 = px.pie(df, names="region", title="区域版图（中国/国际/美国/印度/欧盟/日本）", hole=0.4)
fig3.show()

## 三、62 天时间序列

每天入库多少条？哪个维度在升温？堆积面积图看四维热度演变。

In [6]:
daily = df.groupby(["date", "dimension"]).size().reset_index(name="count")
fig4 = px.area(daily, x="date", y="count", color="dimension",
               line_group="dimension", title="每日入库量 · 四维堆积面积图")
fig4.show()

In [7]:
# 周节奏：星期几最活跃？（0=周一）
df["weekday"] = df["dt"].dt.dayofweek
wd = df.groupby(["weekday", "dimension"]).size().reset_index(name="count")
fig5 = px.bar(wd, x="weekday", y="count", color="dimension",
              title="按星期几的入库节奏（政府源周末是否断更？）")
fig5.show()

## 四、来源贡献分析

哪些源撑起了内容大盘？各维度的头部来源分别是谁？

In [8]:
src_top = df["source"].value_counts().head(15).reset_index()
src_top.columns = ["source", "count"]
fig6 = px.bar(src_top, x="count", y="source", orientation="h",
              title="来源贡献 TOP15（2026-06-22 → 08-19）")
fig6.update_layout(yaxis={"categoryorder": "total ascending"})
fig6.show()

In [9]:
dim_src = (df.groupby(["dimension", "source"]).size()
             .reset_index(name="count")
             .sort_values(["dimension", "count"], ascending=[True, False])
             .groupby("dimension").head(3))
fig7 = px.bar(dim_src, x="count", y="source", color="dimension",
              facet_col="dimension", facet_col_wrap=2, orientation="h",
              title="各维度头部来源 TOP3")
fig7.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig7.show()

## 五、打分体系透视 ★

v2.0 五维模型：内容强度 30 + 来源权威 25 + 主题相关 25 + 人物 10 + 时效 10。
按"得分率"（得分/满分）比较五维的松紧，验证打分是否偏科。

In [10]:
MAXES = {"src_score": 25, "strength": 30, "topic": 25, "people_score": 10, "freshness": 10}
rate = pd.DataFrame({k: df[k] / v for k, v in MAXES.items()})
rate_mean = rate.mean().sort_values().reset_index()
rate_mean.columns = ["维度", "得分率"]
fig8 = px.bar(rate_mean, x="得分率", y="维度", orientation="h",
              title="五维得分率对比（越低 = 该维度越严苛）",
              text=rate_mean["得分率"].map(lambda x: f"{x:.0%}"))
fig8.update_layout(yaxis={"categoryorder": "total ascending"})
fig8.update_traces(textposition="outside")
fig8.show()

In [11]:
# 内容强度（strength）按维度看分布 —— 各维度是否都吃满了自己的 30 分档？
fig9 = px.box(df, x="dimension", y="strength", color="dimension",
              title="内容强度分布（按维度，满分 30）")
fig9.show()

In [12]:
# 来源权威分 vs 最终得分：权威性真的转化为高分了吗？
fig10 = px.scatter(df, x="src_score", y="score", color="dimension",
                   hover_data=["title"], opacity=0.65,
                   title="来源权威分 vs 综合得分（按维度着色）")
fig10.show()

In [13]:
# ⚠️ 库里 freshness 是「首次收录时刻快照」，不随采集更新（老条目永不衰减）
# → 用 published_at 和当前时间重算"真实时效分"，对比出哪些条目虚高
from datetime import datetime, timezone

NOW = datetime.now(timezone.utc)

def real_freshness(published_at):
    dt = pd.to_datetime(published_at, utc=True, errors="coerce")
    if pd.isna(dt):
        return 0
    hours = (NOW - dt).total_seconds() / 3600
    if hours < 0:
        return 10
    if hours < 24:
        return 10
    if hours < 48:
        return 8
    if hours < 72:
        return 6
    if hours < 96:
        return 4
    return 2

df["freshness_real"] = df["published_at"].map(real_freshness)
df["freshness_diff"] = df["freshness"] - df["freshness_real"]  # >0 = 库里虚高

inflated = int((df["freshness_diff"] > 0).sum())
print(f"库里 freshness 虚高的条目: {inflated}/{len(df)}（占 {inflated/len(df):.0%}）")
print("示例（库里 10 分但真实时效已衰减）：")
for _, r in df[df["freshness_diff"] > 0].sort_values("freshness_diff", ascending=False).head(5).iterrows():
    print(f"  [{r['dimension']}] {str(r['title'])[:34]} | 快照{r['freshness']} → 实时{r['freshness_real']}")

fig10b = px.histogram(df, x="freshness_diff", nbins=12,
                      title="快照 freshness − 实时 freshness（>0 = 库里虚高）")
fig10b.show()

库里 freshness 虚高的条目: 133/504（占 26%）
示例（库里 10 分但真实时效已衰减）：
  [行业] 印度民航监管方拟强制国际航线航司报告至少90%年度运营排放,为COR | 快照10 → 实时8
  [行业] 澳大利亚政府决定分阶段废止联邦 "Climate Active" 自 | 快照10 → 实时8
  [金融] 欧委会批准马耳他6000万欧元社会气候计划,ETS2建筑交通2028 | 快照10 → 实时8
  [行业] 比亚迪7月全球销42万辆同比增22%出口占43%,电车链脱碳倒逼钢铁 | 快照10 → 实时8
  [政府] 马来西亚更新国家能效行动计划(NEEAP 2.0,2026-2035 | 快照10 → 实时8


## 六、关键词透视

jieba 分词，看四个维度各自在聊什么话题（标题+摘要前 200 字）。

In [14]:
STOP = set(("的 了 是 在 和 与 及 等 中 为 将 对 并 由 通过 表示 称 指出 记者 报道 相关 进行 以及 我们 他们 其 该 这 那 有 也 就 都 而 但 或 一个 没有 不是 被 把 从 向 到 于 与 年 月 日 个 上 下 后 前 内 外 说 称 显示 据 据悉 目前 近日 今天 昨日 新 大 小 更 最 已 能 会 要 可 以 及 为 各 每 多 少 万 亿 元 人 公司 中国 美国 欧洲 日本 印度 政府 国际 全球 the a an and of to in for on with is are was were be by as at or from this that it its not have has had").split())

def top_words(texts, n=14):
    c = Counter()
    for t in texts:
        for w in jieba.lcut(str(t)[:200]):
            w = w.strip()
            if len(w) < 2 or w in STOP or w.isdigit():
                continue
            c[w] += 1
    return c.most_common(n)

rows = []
for dim, grp in df.groupby("dimension"):
    for w, n in top_words(grp["title"].tolist() + grp["summary"].fillna("").tolist()):
        rows.append({"dimension": dim, "word": w, "count": n})
kw = pd.DataFrame(rows)

fig11 = px.bar(kw, x="count", y="word", color="dimension",
               facet_row="dimension", orientation="h",
               title="四维关键词 TOP14")
fig11.update_layout(showlegend=False, height=1200)
fig11.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig11.show()

Building prefix dict from the default dictionary ...


Loading model from cache /tmp/jieba.cache


Loading model cost 0.549 seconds.


Prefix dict has been built successfully.


## 七、高分回顾（S + A 级）

62 天里最重要的政策动态清单，可直接转发或写入 wiki。

In [15]:
top = (df[df["score_level"].isin(["S", "A"])]
       .sort_values("score", ascending=False)
       .reset_index(drop=True))
print(f"S/A 级共 {len(top)} 条，最高分 {top['score'].max()}")

fig12 = go.Figure(go.Table(
    header=dict(values=["#", "标题", "维度", "来源", "分数", "链接"],
                fill_color="#0052ff", font=dict(color="white"), align="left"),
    cells=dict(
        values=[top.index + 1,
                top["title"].str[:38],
                top["dimension"],
                top["source"],
                top["score"],
                top["url"].str[:45]],
        align="left", height=28)))
fig12.update_layout(title="S/A 级条目清单（62 天精华）", height=60 + 30 * len(top))
fig12.show()

S/A 级共 46 条，最高分 85


## 八、结论与洞察

自动汇总关键数字，下面留了 markdown 区给你写策展心得~

In [16]:
print("=" * 46)
print("📊 62 天数据速览")
print("=" * 46)
print(f"总条数          : {len(df)}")
print(f"四维分布        : {df['dimension'].value_counts().to_dict()}")
print(f"S/A 级占比      : {(df['score_level'].isin(['S','A'])).mean():.1%}")
print(f"平均分          : {df['score'].mean():.1f}  |  中位数 {df['score'].median():.0f}")
print(f"覆盖天数        : {df['date'].nunique()} 天，日均 {len(df)/df['date'].nunique():.1f} 条")
print(f"五维得分率(低=严): {dict(rate.mean().sort_values().round(2))}")
print(f"来源 TOP5       : {df['source'].value_counts().head(5).to_dict()}")

📊 62 天数据速览
总条数          : 504
四维分布        : {'政府': 194, 'AI': 148, '行业': 125, '金融': 37}
S/A 级占比      : 9.1%
平均分          : 51.5  |  中位数 49
覆盖天数        : 17 天，日均 29.6 条
五维得分率(低=严): {'people_score': np.float64(0.01), 'topic': np.float64(0.47), 'strength': np.float64(0.48), 'freshness': np.float64(0.53), 'src_score': np.float64(0.8)}
来源 TOP5       : {'中国能源报': 58, '欧盟委员会': 30, '中国人民银行': 30, '中国碳交易网': 29, '美国DOE': 29}


### 📝 我的策展笔记

（在这里写下你的观察：哪个维度信号最强、哪些高分条目值得进 wiki、下周关注什么……）